In [125]:
import pandas as pd

df = pd.read_csv('language.csv')




#fetch the first 5 rows

df.head()

,Code,Name
0,af,Afrikaans
1,af_ZA,Afrikaans (South Africa)
2,ar,Arabic
3,ar_AE,Arabic (U.A.E.)
4,ar_BH,Arabic (Bahrain)


In [126]:
## fetch the last 5 rows

df.tail()

,Code,Name
234,zh_MO,Chinese (Macau)
235,zh_SG,Chinese (Singapore)
236,zh_TW,Chinese (T)
237,zu,Zulu
238,zu_ZA,Zulu (South Africa)


In [127]:
df.describe()

,Code,Name
count,239,239
unique,228,239
top,se_FI,Afrikaans
freq,3,1


In [128]:
df.dtypes

Code    str
Name    str
dtype: object

In [129]:
#handling missing values


df.isnull()

,Code,Name
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False
...,...,...
234,False,False
235,False,False
236,False,False
237,False,False


In [130]:
df.loc[[2,3,4],'i_know'] = False

In [131]:
df

,Code,Name,i_know
0,af,Afrikaans,NaN
1,af_ZA,Afrikaans (South Africa),NaN
2,ar,Arabic,False
3,ar_AE,Arabic (U.A.E.),False
4,ar_BH,Arabic (Bahrain),False
...,...,...,...
234,zh_MO,Chinese (Macau),NaN
235,zh_SG,Chinese (Singapore),NaN
236,zh_TW,Chinese (T),NaN
237,zu,Zulu,NaN


In [132]:
# df.isnull().any()
df.isnull().any(axis=1)

0       True
1       True
2      False
3      False
4      False
       ...  
234     True
235     True
236     True
237     True
238     True
Length: 239, dtype: bool

In [133]:
df.isnull().sum()

Code        0
Name        0
i_know    236
dtype: int64

In [134]:
df['i_know_fill']= df['i_know'].fillna(df['i_know'].mode()[0]) # use mode if bool else use mean median if integer

df

,Code,Name,i_know,i_know_fill
0,af,Afrikaans,NaN,False
1,af_ZA,Afrikaans (South Africa),NaN,False
2,ar,Arabic,False,False
3,ar_AE,Arabic (U.A.E.),False,False
4,ar_BH,Arabic (Bahrain),False,False
...,...,...,...,...
234,zh_MO,Chinese (Macau),NaN,False
235,zh_SG,Chinese (Singapore),NaN,False
236,zh_TW,Chinese (T),NaN,False
237,zu,Zulu,NaN,False


In [135]:
df = df.rename(columns={'Name': 'Country-Name'})
df

,Code,Country-Name,i_know,i_know_fill
0,af,Afrikaans,NaN,False
1,af_ZA,Afrikaans (South Africa),NaN,False
2,ar,Arabic,False,False
3,ar_AE,Arabic (U.A.E.),False,False
4,ar_BH,Arabic (Bahrain),False,False
...,...,...,...,...
234,zh_MO,Chinese (Macau),NaN,False
235,zh_SG,Chinese (Singapore),NaN,False
236,zh_TW,Chinese (T),NaN,False
237,zu,Zulu,NaN,False


In [136]:
df = df.rename(columns={'i_know':'speakers'})

In [137]:
df['speakers'] = df['speakers'].fillna(False).astype('int32')



In [138]:

df

,Code,Country-Name,speakers,i_know_fill
0,af,Afrikaans,0,False
1,af_ZA,Afrikaans (South Africa),0,False
2,ar,Arabic,0,False
3,ar_AE,Arabic (U.A.E.),0,False
4,ar_BH,Arabic (Bahrain),0,False
...,...,...,...,...
234,zh_MO,Chinese (Macau),0,False
235,zh_SG,Chinese (Singapore),0,False
236,zh_TW,Chinese (T),0,False
237,zu,Zulu,0,False


In [139]:
df

,Code,Country-Name,speakers,i_know_fill
0,af,Afrikaans,0,False
1,af_ZA,Afrikaans (South Africa),0,False
2,ar,Arabic,0,False
3,ar_AE,Arabic (U.A.E.),0,False
4,ar_BH,Arabic (Bahrain),0,False
...,...,...,...,...
234,zh_MO,Chinese (Macau),0,False
235,zh_SG,Chinese (Singapore),0,False
236,zh_TW,Chinese (T),0,False
237,zu,Zulu,0,False


In [142]:
# df = df.drop(columns='i_know_fill',inplace=True) # dont do this this will drop your whole csv this is buggy

df = df.drop('i_know_fill',axis=1) 

In [ ]:
## vibe coded part to fetch the realtime speaker data



import pandas as pd
import requests
from io import StringIO
from difflib import get_close_matches

# ---------- 1) Scrape + clean ----------
url = "https://en.wikipedia.org/wiki/List_of_languages_by_total_number_of_speakers"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
tables = pd.read_html(StringIO(html))

real = tables[0].copy()
real.columns = [c[0] if c[0] == c[1] else f"{c[0]} | {c[1]}" for c in real.columns]
total_col = [c for c in real.columns if "Millions" in c][-1]

real = real[["Language", total_col]].copy()
real.columns = ["wiki_lang", "speakers"]
real["wiki_lang"] = real["wiki_lang"].str.replace(r"\[.*?\]", "", regex=True).str.strip()
real["speakers"] = pd.to_numeric(
    real["speakers"].astype(str).str.replace(r"[^\d.]", "", regex=True),
    errors="coerce") * 1_000_000
real = real.dropna()

# ---------- 2) Base language nikal ----------
df = df.drop(columns=["speakers", "base_lang"], errors="ignore")
df["base_lang"] = (df["Country-Name"]
                   .str.replace(r"\s*\(.*\)", "", regex=True)
                   .str.strip())

# ---------- 3) Fuzzy matching ----------
wiki_names = real["wiki_lang"].tolist()
manual = {"Chinese": "Mandarin Chinese"}          # jo fuzzy se na pakde, yahan daal

def match(name):
    if name in manual:
        return manual[name]
    # wiki naam me tere naam ka hona (jaise "Standard Arabic" me "Arabic")
    contains = [w for w in wiki_names if name.lower() in w.lower()]
    if contains:
        return contains[0]
    close = get_close_matches(name, wiki_names, n=1, cutoff=0.85)
    return close[0] if close else None

df["wiki_lang"] = df["base_lang"].apply(match)

# ---------- 4) Merge ----------
df = df.merge(real, on="wiki_lang", how="left")
df["speakers"] = df["speakers"].fillna(0).astype("int64")

# ---------- 5) Check ----------
print("Matched rows  :", (df["speakers"] > 0).sum())
print("Unmatched rows:", (df["speakers"] == 0).sum())
print(df[["Country-Name", "wiki_lang", "speakers"]].query("speakers > 0").head(15))

Matched rows  : 105
Unmatched rows: 134
             Country-Name                                wiki_lang   speakers
2                  Arabic  Modern Standard Arabic (excl. dialects)  335000000
3         Arabic (U.A.E.)  Modern Standard Arabic (excl. dialects)  335000000
4        Arabic (Bahrain)  Modern Standard Arabic (excl. dialects)  335000000
5        Arabic (Algeria)  Modern Standard Arabic (excl. dialects)  335000000
6          Arabic (Egypt)  Modern Standard Arabic (excl. dialects)  335000000
7           Arabic (Iraq)  Modern Standard Arabic (excl. dialects)  335000000
8         Arabic (Jordan)  Modern Standard Arabic (excl. dialects)  335000000
9         Arabic (Kuwait)  Modern Standard Arabic (excl. dialects)  335000000
10       Arabic (Lebanon)  Modern Standard Arabic (excl. dialects)  335000000
11         Arabic (Libya)  Modern Standard Arabic (excl. dialects)  335000000
12       Arabic (Morocco)  Modern Standard Arabic (excl. dialects)  335000000
13          Arabic (Oman

In [151]:


df


,Code,Country-Name,base_lang,wiki_lang,speakers
0,af,Afrikaans,Afrikaans,NaN,0
1,af_ZA,Afrikaans (South Africa),Afrikaans,NaN,0
2,ar,Arabic,Arabic,Modern Standard Arabic (excl. dialects),335000000
3,ar_AE,Arabic (U.A.E.),Arabic,Modern Standard Arabic (excl. dialects),335000000
4,ar_BH,Arabic (Bahrain),Arabic,Modern Standard Arabic (excl. dialects),335000000
...,...,...,...,...,...
234,zh_MO,Chinese (Macau),Chinese,Mandarin Chinese,0
235,zh_SG,Chinese (Singapore),Chinese,Mandarin Chinese,0
236,zh_TW,Chinese (T),Chinese,Mandarin Chinese,0
237,zu,Zulu,Zulu,NaN,0


In [148]:
df

,Code,Country-Name,base_lang,speakers
0,af,Afrikaans,Afrikaans,0
1,af_ZA,Afrikaans (South Africa),Afrikaans,0
2,ar,Arabic,Arabic,0
3,ar_AE,Arabic (U.A.E.),Arabic,0
4,ar_BH,Arabic (Bahrain),Arabic,0
...,...,...,...,...
234,zh_MO,Chinese (Macau),Mandarin Chinese,0
235,zh_SG,Chinese (Singapore),Mandarin Chinese,0
236,zh_TW,Chinese (T),Mandarin Chinese,0
237,zu,Zulu,Zulu,0
